# 4-4 DataFrame 绘图

DataFrame 适合在同一张图中比较多列数据。
默认使用行索引作为横轴，每个数值列对应一组数据，列名通常会自动成为图例。

本节学习多列绘图、选择某行或某列绘图，以及分组、堆积和分布图。

> 约定：使用 `np` 作为 NumPy 的别名，使用 `plt` 作为 `matplotlib.pyplot` 的别名。
> 下方准备单元格会选择可用的中文字体；可以从项目根目录或本章目录运行。
> `fonts` 文件夹是可选的，没有该文件夹时会查找系统字体。
> 如果没有可用的中文字体，会显示提示并使用默认字体继续绘图，中文可能显示为方框。
> 字体设置只作用于当前 Python 进程，详细原理见第 4-6 节。

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

# fonts 文件夹是可选的；没有字体文件时，使用系统已安装的字体。
project_dir = Path('.')
if Path.cwd().name == '第4章-Matplotlib':
    project_dir = Path('..')
font_files = [
    project_dir / 'fonts' / 'NotoSansCJKsc-Regular.otf',
    project_dir / 'fonts' / 'NotoSansSC-Regular.ttf',
]
font_path = next((path for path in font_files if path.is_file()), None)

if font_path is None:
    font_candidates = [
        'Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'Noto Sans SC',
        'Source Han Sans SC', 'PingFang SC', 'Heiti SC', 'Songti SC',
        'Arial Unicode MS',
    ]
    available_fonts = {entry.name for entry in font_manager.fontManager.ttflist}
    font_name = next((name for name in font_candidates if name in available_fonts), None)
    if font_name is None:
        font_name = 'DejaVu Sans'
        print('未找到中文字体，暂用 DejaVu Sans，图中的中文可能显示为方框。')
        print('可安装 Noto Sans CJK SC，或将中文字体文件放入项目的 fonts 文件夹后重新运行。')
    font_path = Path(font_manager.findfont(
        font_manager.FontProperties(family=font_name), fallback_to_default=False
    ))
else:
    font_name = font_manager.FontProperties(fname=str(font_path)).get_name()

font_manager.fontManager.addfont(str(font_path))
plt.rcParams['font.family'] = [font_name]
plt.rcParams['axes.unicode_minus'] = False

import pandas as pd

## 1. 准备数据

创建 10 行、3 列的整数数据。`rng.integers(1, 15, size=(10, 3))` 的取值范围为 `[1, 15)`。

In [ ]:
rng = np.random.default_rng(42)
df = pd.DataFrame(rng.integers(1, 15, size=(10, 3)), columns=['A', 'B', 'C'])
df.index.name = '样本序号'
df.head()

## 2. 折线图

### 2.1 同时绘制多列

`df.plot(kind='line')` 为 `A`、`B`、`C` 三列分别绘制一条折线。

In [ ]:
ax = df.plot(kind='line', marker='o', figsize=(7, 4), title='三列数据的变化')
ax.set_xlabel('样本序号')
ax.set_ylabel('数值')
plt.tight_layout()
plt.show()

### 2.2 选择一列绘图

`df['A']` 返回 Series；`df.A` 也能用于这里的列名。
方括号写法更通用，可以处理包含空格或与对象属性同名的列。

In [ ]:
ax = df['A'].plot(kind='line', marker='o', figsize=(7, 4), title='A 列数据')
ax.set_xlabel('样本序号')
ax.set_ylabel('A 列数值')
plt.tight_layout()
plt.show()

### 2.3 选择一行绘图

`df.loc[0]` 选择行标签为 `0` 的一行，得到以列名 `A`、`B`、`C` 为索引的 Series。
此时横轴变为列名，纵轴为这一行的三个数值。

下面用折线图观察选择一行后的结果；当列名仅表示无顺序类别时，条形图通常更合适。

In [ ]:
ax = df.loc[0].plot(kind='line', marker='o', figsize=(6, 4), title='第 0 行的三个数值')
ax.set_xlabel('列名')
ax.set_ylabel('数值')
plt.tight_layout()
plt.show()

## 3. 分组条形图

### 3.1 竖向条形图

`kind='bar'` 把每个行索引作为一组，组内并列显示 `A`、`B`、`C` 三列的数值。

In [ ]:
ax = df.plot(kind='bar', figsize=(8, 4), rot=0, title='三列数据的分组条形图')
ax.set_xlabel('样本序号')
ax.set_ylabel('数值')
plt.tight_layout()
plt.show()

### 3.2 横向条形图

`kind='barh'` 使用横向条形，纵轴是行索引，横轴是数值。

In [ ]:
ax = df.plot(kind='barh', figsize=(7, 6), title='三列数据的横向分组条形图')
ax.set_xlabel('数值')
ax.set_ylabel('样本序号')
plt.tight_layout()
plt.show()

## 4. 堆积条形图

`stacked=True` 将同一行中各列的条形叠在一起。本例都是正数，所以总高度等于这一行的总和。
分组条形图便于比较各列的数值，堆积条形图便于同时观察总量和组成。

In [ ]:
ax = df.plot(kind='bar', stacked=True, figsize=(8, 4), rot=0, title='三列数据的堆积条形图')
ax.set_xlabel('样本序号')
ax.set_ylabel('数值合计')
plt.tight_layout()
plt.show()

## 5. 面积图

`kind='area'` 绘制面积图，默认堆积显示。每一层的厚度对应一列的值，
上边界显示各列累计总量。

> 面积图不会自动把数值转换成百分比；原始数据绘制出来的仍是原始数量。
> 堆积面积图要求每列数据整体非负或整体非正，本例均为正数。

In [ ]:
ax = df.plot(kind='area', figsize=(7, 4), title='三列数据的堆积面积图')
ax.set_xlabel('样本序号')
ax.set_ylabel('数值合计')
plt.tight_layout()
plt.show()

如果希望观察占比，需要先按行除以该行总和。本例的行总和都大于 `0`。
`axis=0` 表示让行总和按 DataFrame 的行标签对齐。

In [ ]:
proportions = df.div(df.sum(axis=1), axis=0)
ax = proportions.plot(kind='area', figsize=(7, 4), title='各列在每行中的占比')
ax.set_xlabel('样本序号')
ax.set_ylabel('占比')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 6. 直方图

### 6.1 多列直方图

`df.plot(kind='hist')` 分别统计各列的分布，再叠放到同一坐标系中。
设置 `alpha` 降低不透明度，可以减少互相遮挡；所有列使用相同分箱便于比较。

In [ ]:
bin_edges = np.arange(0.5, 15.5, 1)
ax = df.plot(kind='hist', bins=bin_edges, alpha=0.5,
             figsize=(7, 4), title='A、B、C 三列的数值分布')
ax.set_xlabel('数值')
ax.set_ylabel('频数')
plt.tight_layout()
plt.show()

### 6.2 单列直方图

如果多列互相遮挡，也可以先选择一列再绘图。

In [ ]:
ax = df['A'].plot(kind='hist', bins=bin_edges, rwidth=0.8,
                  figsize=(7, 4), title='A 列的数值分布')
ax.set_xlabel('A 列数值')
ax.set_ylabel('频数')
plt.tight_layout()
plt.show()

## 7. 要点总结

- DataFrame 折线图默认以行索引为横轴，每个数值列各绘制一条曲线。
- `df['A']` 选择列，`df.loc[0]` 按标签选择行；选择后可以继续使用 Series 绘图。
- `bar` 和 `barh` 绘制分组条形图，`stacked=True` 改为堆积显示。
- 面积图默认堆积，展示的是输入值；要表示占比，需要先归一化。
- 多列直方图可使用相同分箱与透明度，单列直方图更便于单独检查分布。